In [1]:
import os, json, time, copy, pickle
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg'); import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

BASE   = r'D:\FYP\Eco-Twin'
V2_DIR = os.path.join(BASE, 'v2_clean')
OUT    = os.path.join(V2_DIR, 'ecotwin_outputs')
SCALER_DIR, PLOT_DIR   = os.path.join(OUT, 'scalers'), os.path.join(OUT, 'plots')
MODEL_DIR, RESULTS_DIR = os.path.join(OUT, 'models'),  os.path.join(OUT, 'results')

DATASET_CSV     = os.path.join(V2_DIR, 'EcoTwin_Final_Dataset.csv')
STATIONS_EXCEL  = os.path.join(V2_DIR, 'Delhi_list_v2.xlsx')
ERA5_WIND_NPY   = os.path.join(V2_DIR, 'openmeteo_reanalysis_uv.npy')
DYN_ADJ_OM_NPY  = os.path.join(V2_DIR, 'dynamic_adj_om.npy')
STATIC_ADJ_NPY  = os.path.join(V2_DIR, 'static_adj_matrix.npy')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
INPUT_WINDOW, HORIZON = 48, 24
BATCH_SIZE = 16
LOSS_VOL_ALPHA = 1.0
SEEDS = [42, 7, 123, 2024, 999]

cfg = json.load(open(os.path.join(MODEL_DIR, 'v3_config.json')))
FEATURE_COLS = cfg['feature_cols']; F = cfg['in_features']
HIDDEN_DIM, GRU_LAYERS = cfg['hidden_dim'], cfg['gru_layers']
ATTN_HEADS, DROPOUT, GCN_HOPS = cfg['attn_heads'], cfg['dropout'], cfg['gcn_hops']
N_FUT = cfg['n_fut']
print(f'Device {DEVICE} | F={F} | n_fut={N_FUT} | seeds {SEEDS}')

Device cuda | F=26 | n_fut=10 | seeds [42, 7, 123, 2024, 999]


In [2]:
df = pd.read_csv(DATASET_CSV, parse_dates=['Timestamp'])
df['Station'] = df['Station'].str.strip()
df = df.sort_values(['Station', 'Timestamp']).reset_index(drop=True)
df_st = pd.read_excel(STATIONS_EXCEL)
df_st.columns = df_st.columns.str.strip().str.lower()
df_st['station'] = df_st['station'].str.strip()
station_names = df_st['station'].tolist()
N = len(station_names)

timestamps = np.array(sorted(df['Timestamp'].unique())); T = len(timestamps)
era5 = np.load(ERA5_WIND_NPY).astype(np.float32)
era5_spd = np.hypot(era5[..., 0], era5[..., 1])
dyn_adj = np.load(DYN_ADJ_OM_NPY).astype(np.float32)                 # A(t) from ERA5 wind (T,N,N)
static_adj = np.load(STATIC_ADJ_NPY).astype(np.float32)
np.fill_diagonal(static_adj, 0.0)
assert dyn_adj.shape == (T, N, N)
print(f'T={T}  dyn_adj(ERA5) {dyn_adj.shape}  static edges {int((static_adj>0).sum())}')

T=19921  dyn_adj(ERA5) (19921, 37, 37)  static edges 508


In [3]:
BASE_FEATS = ['pm2.5','pm10','no2','so2','co','o3','rh','at']
g = df.groupby('Station', sort=False)
df['pm_roll6_mean']  = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).mean())
df['pm_roll6_std']   = g['pm2.5'].transform(lambda s: s.rolling(6,  min_periods=1).std()).fillna(0.0)
df['pm_roll24_mean'] = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).mean())
df['pm_roll24_std']  = g['pm2.5'].transform(lambda s: s.rolling(24, min_periods=1).std()).fillna(0.0)
df['pm_delta3']      = g['pm2.5'].transform(lambda s: s - s.shift(3)).fillna(0.0)
tsg = df.groupby('Timestamp', sort=False)['pm2.5']
df['city_mean_pm']  = tsg.transform('mean')
df['city_max_pm']   = tsg.transform('max')
df['pm_minus_city'] = df['pm2.5'] - df['city_mean_pm']
hh = df['Timestamp'].dt.hour; dow = df['Timestamp'].dt.dayofweek; mon = df['Timestamp'].dt.month
df['hour_sin'] = np.sin(2*np.pi*hh/24);      df['hour_cos'] = np.cos(2*np.pi*hh/24)
df['dow_sin']  = np.sin(2*np.pi*dow/7);      df['dow_cos']  = np.cos(2*np.pi*dow/7)
df['mon_sin']  = np.sin(2*np.pi*(mon-1)/12); df['mon_cos']  = np.cos(2*np.pi*(mon-1)/12)
df['is_weekend'] = (dow >= 5).astype(float)

CPCB_COLS = (BASE_FEATS +
    ['pm_roll6_mean','pm_roll6_std','pm_roll24_mean','pm_roll24_std','pm_delta3',
     'city_mean_pm','city_max_pm','pm_minus_city',
     'hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend'])
Xc = np.zeros((T, N, len(CPCB_COLS)), dtype=np.float32)
for fi, col in enumerate(CPCB_COLS):
    wide = df.pivot_table(index='Timestamp', columns='Station', values=col).reindex(columns=station_names)
    Xc[:, :, fi] = np.nan_to_num(wide.values.astype(np.float32))

built = (BASE_FEATS + ['u_wind', 'v_wind', 'wspd'] + CPCB_COLS[len(BASE_FEATS):])
assert built == FEATURE_COLS, 'feature order mismatch vs v3_config'
X_raw = np.concatenate([
    Xc[:, :, :len(BASE_FEATS)],
    era5[:, :, 0:1], era5[:, :, 1:2], era5_spd[:, :, None],
    Xc[:, :, len(BASE_FEATS):],
], axis=2).astype(np.float32)
Y_raw = X_raw[:, :, 0].copy()

TIME_COLS = ['hour_sin','hour_cos','dow_sin','dow_cos','mon_sin','mon_cos','is_weekend']
TIME_IDX  = [FEATURE_COLS.index(c) for c in TIME_COLS]
U_IDX, V_IDX, W_IDX = FEATURE_COLS.index('u_wind'), FEATURE_COLS.index('v_wind'), FEATURE_COLS.index('wspd')
FUT_IDX   = TIME_IDX + [U_IDX, V_IDX, W_IDX]
print(f'F={X_raw.shape[2]}  NaNs {int(np.isnan(X_raw).sum())}')

F=26  NaNs 0


In [4]:
feat_scaler = pickle.load(open(os.path.join(SCALER_DIR, 'v3_feature_scaler.pkl'), 'rb'))
log_scaler  = pickle.load(open(os.path.join(SCALER_DIR, 'v3_log_scaler.pkl'), 'rb'))
Xs_full = feat_scaler.transform(X_raw.reshape(-1, F)).reshape(T, N, F).astype(np.float32)
def from_log(a): return np.expm1(log_scaler.inverse_transform(a.reshape(-1, 1)).reshape(a.shape))
def to_log(a):   return log_scaler.transform(np.log1p(np.clip(a, 0, None)).reshape(-1, 1)).reshape(a.shape).astype(np.float32)
Yl_full = to_log(Y_raw)

train_end_idx = int(T * 0.70); val_end_idx = int(T * 0.85)
def split_starts(lo, hi):
    return np.arange(lo, lo + (hi - lo) - (INPUT_WINDOW + HORIZON) + 1)
train_starts = split_starts(0, train_end_idx)
val_starts   = split_starts(train_end_idx, val_end_idx)
test_starts  = split_starts(val_end_idx, T)
del X_raw
print(f'windows -- train {len(train_starts):,} | val {len(val_starts):,} | test {len(test_starts):,}')

windows -- train 13,873 | val 2,917 | test 2,918


In [5]:
class DS(Dataset):
    def __init__(self, starts, weights=None):
        self.s = np.asarray(starts, np.int64)
        self.w = (weights if weights is not None else np.ones(len(self.s), np.float32)).astype(np.float32)
    def __len__(self):
        return len(self.s)
    def __getitem__(self, k):
        g = int(self.s[k]); i1 = g + INPUT_WINDOW; i2 = i1 + HORIZON
        return (torch.from_numpy(Xs_full[g:i1].copy()),
                torch.from_numpy(dyn_adj[g:i1].copy()),
                torch.from_numpy(dyn_adj[i1:i2].copy()),
                torch.from_numpy(np.ascontiguousarray(Xs_full[i1:i2][:, :, FUT_IDX])),
                torch.from_numpy(Yl_full[i1-1].copy()),
                torch.from_numpy(Yl_full[i1:i2].copy()),
                torch.tensor(self.w[k], dtype=torch.float32))

def window_swing(starts):
    return np.array([(Y_raw[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].max(0) -
                      Y_raw[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].min(0)).mean()
                     for g in starts], dtype=np.float32)
tr_sw = window_swing(train_starts); med = float(np.median(tr_sw))
tr_w = np.clip(1.0 + LOSS_VOL_ALPHA * (tr_sw / (med + 1e-6)), 0.5, 4.0).astype(np.float32)
tr_w *= len(tr_w) / tr_w.sum()

tr_ld = DataLoader(DS(train_starts, tr_w), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
va_ld = DataLoader(DS(val_starts),         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
te_ld = DataLoader(DS(test_starts),        batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'batches -- train {len(tr_ld)} | val {len(va_ld)} | test {len(te_ld)}')

batches -- train 868 | val 183 | test 183


In [6]:
class WATGCNv3Ablate(nn.Module):
    def __init__(self, in_features, hidden_dim, gru_layers, attn_heads, forecast_horizon,
                 n_stations, dropout=0.2, n_fut=10, gcn_hops=2, graph_mode='dynamic', static_adj=None):
        super().__init__()
        H = hidden_dim
        self.input_proj = nn.Linear(in_features, H)
        self.gcn_lin  = nn.ModuleList(nn.Linear(H, H) for _ in range(gcn_hops))
        self.gcn_norm = nn.ModuleList(nn.LayerNorm(H) for _ in range(gcn_hops))
        self.act = nn.ReLU()
        self.gru = nn.GRU(H, H, gru_layers, batch_first=True, dropout=dropout if gru_layers > 1 else 0.0)
        self.attn = nn.MultiheadAttention(H, attn_heads, dropout=dropout, batch_first=True)
        self.attn_norm = nn.LayerNorm(H)
        self.dropout = nn.Dropout(dropout)
        self.fut_enc = nn.Sequential(nn.Linear(n_fut, H), nn.ReLU(), nn.Linear(H, H))
        self.decoder = nn.Sequential(nn.Linear(H*3, H), nn.ReLU(), nn.Dropout(dropout), nn.Linear(H, 1))
        self.anchor_gate = nn.Parameter(torch.linspace(1.0, 0.3, forecast_horizon))
        self.H, self.FH, self.graph_mode = H, forecast_horizon, graph_mode
        sa = torch.eye(n_stations) if static_adj is None else torch.from_numpy(static_adj.astype('float32'))
        self.register_buffer('static_adj', sa, persistent=False)

    @staticmethod
    def _n(A):
        return A / A.sum(-1, keepdim=True).clamp(min=1e-6)

    def forward(self, X, At, Afc, Fut, Anchor):
        B, Tin, n, _ = X.shape; dev = X.device; I = torch.eye(n, device=dev)
        h = self.act(self.input_proj(X))
        if self.graph_mode == 'none':
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(h))) + h
        else:
            if self.graph_mode == 'dynamic':
                A = self._n(At + I.view(1, 1, n, n))
            else:
                A = self._n((self.static_adj + I).view(1, 1, n, n)).expand(B, Tin, n, n)
            for lin, nrm in zip(self.gcn_lin, self.gcn_norm):
                h = self.act(nrm(lin(torch.matmul(A, h)))) + h
        hf = h.permute(0, 2, 1, 3).reshape(B*n, Tin, self.H)
        gg, _ = self.gru(hf); gg = self.dropout(gg)
        aa, _ = self.attn(gg, gg, gg)
        ctx = self.attn_norm(aa + gg)[:, -1, :].reshape(B, n, self.H)
        ctx_rep = ctx.unsqueeze(1).expand(B, self.FH, n, self.H)
        if self.graph_mode == 'none':
            ctx_fc = ctx_rep
        elif self.graph_mode == 'dynamic':
            ctx_fc = torch.matmul(self._n(Afc + I.view(1, 1, n, n)), ctx_rep)
        else:
            ctx_fc = torch.matmul(self._n((self.static_adj + I).view(1, 1, n, n)), ctx_rep)
        fut_emb = self.fut_enc(Fut)
        resid = self.decoder(torch.cat([ctx_rep, ctx_fc, fut_emb], dim=-1)).squeeze(-1)
        return resid + Anchor.unsqueeze(1) * self.anchor_gate.view(1, self.FH, 1)

SA = static_adj.copy()
def build(mode):
    return WATGCNv3Ablate(F, HIDDEN_DIM, GRU_LAYERS, ATTN_HEADS, HORIZON, N, DROPOUT,
                          N_FUT, GCN_HOPS, graph_mode=mode, static_adj=SA).to(DEVICE)
for mode in ['dynamic', 'static', 'none']:
    _m = build(mode); print(f'{mode:8s} {sum(p.numel() for p in _m.parameters()):,} params'); del _m

dynamic  18,297 params
static   18,297 params
none     18,297 params


In [7]:
def weighted_huber(pred, target, w, delta=1.0):
    absr = (pred - target).abs(); quad = torch.clamp(absr, max=delta)
    per = (0.5*quad**2 + delta*(absr - quad)).mean(dim=(1, 2))
    return (per * w).mean() / w.mean().clamp(min=1e-6)

def run_epoch(model, loader, opt=None, sched=None):
    tr = opt is not None; model.train(tr); tot = n = 0
    for X, At, Afc, Fut, anc, y, w in loader:
        X, At, Afc = X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE)
        Fut, anc, y, w = Fut.to(DEVICE), anc.to(DEVICE), y.to(DEVICE), w.to(DEVICE)
        with torch.set_grad_enabled(tr):
            out = model(X, At, Afc, Fut, anc)
            loss = weighted_huber(out, y, w if tr else torch.ones_like(w))
        if tr:
            opt.zero_grad(); loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            if sched: sched.step()
        tot += loss.item() * X.size(0); n += X.size(0)
    return tot / n

@torch.no_grad()
def predict(model, loader):
    model.eval(); P, A = [], []
    for X, At, Afc, Fut, anc, y, w in loader:
        P.append(model(X.to(DEVICE), At.to(DEVICE), Afc.to(DEVICE), Fut.to(DEVICE), anc.to(DEVICE)).cpu().numpy())
        A.append(y.numpy())
    return np.concatenate(P), np.concatenate(A)

def sc(pl, al):
    pr, ar = from_log(pl), from_log(al)
    return dict(rmse=float(np.sqrt(mean_squared_error(ar.ravel(), pr.ravel()))),
                mae=float(mean_absolute_error(ar.ravel(), pr.ravel())),
                r2=float(r2_score(ar.ravel(), pr.ravel())))

def train_one(mode, seed, max_epochs=60, patience=12, lr=1e-3):
    torch.manual_seed(seed); np.random.seed(seed)
    model = build(mode)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    steps = len(tr_ld) * max_epochs; warm = max(1, int(0.05 * steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: s/warm if s < warm else 0.5*(1 + np.cos(np.pi*(s-warm)/max(1, steps-warm))))
    best, best_state, ctr = 1e9, None, 0
    for ep in range(1, max_epochs+1):
        t0 = time.time()
        trl = run_epoch(model, tr_ld, opt, sched); vl = run_epoch(model, va_ld)
        if vl < best - 1e-5: best, best_state, ctr = vl, copy.deepcopy(model.state_dict()), 0
        else: ctr += 1
        if ep % 4 == 0 or ctr >= patience:
            print(f'    ep {ep:>2} train {trl:.4f} val {vl:.4f} ({time.time()-t0:.0f}s) best {best:.4f}')
        if ctr >= patience: break
    model.load_state_dict(best_state)
    return model
print('ready')

ready


In [8]:
test_pred = {}; test_actual = None

# dynamic = WA-TGCN v3 : reuse trained checkpoints, no retrain
dyn_preds = []
for sd in SEEDS:
    m = build('dynamic')
    m.load_state_dict(torch.load(os.path.join(MODEL_DIR, f'watgcn_v3_seed{sd}.pt'), map_location=DEVICE))
    p, a = predict(m, te_ld); dyn_preds.append(p); test_actual = a
    del m
test_pred['WA-TGCN v3 (dynamic)'] = np.mean(dyn_preds, axis=0)
print(f"dynamic  test RMSE {sc(test_pred['WA-TGCN v3 (dynamic)'], test_actual)['rmse']:.2f}  (nb14 7-seed was 58.79)")

# static-GCN and no-graph : train 5 seeds each
for mode, label in [('static', 'Static-GCN v3'), ('none', 'No-graph v3')]:
    seed_preds = []
    for sd in SEEDS:
        print(f'\n[{label}] seed {sd}')
        m = train_one(mode, sd)
        p, a = predict(m, te_ld); seed_preds.append(p)
        print(f'  -> seed {sd} test RMSE {sc(p, a)["rmse"]:.2f}')
        del m; torch.cuda.empty_cache()
    test_pred[label] = np.mean(seed_preds, axis=0)

# persistence
anc_all = np.stack([Yl_full[g+INPUT_WINDOW-1] for g in test_starts])
test_pred['Persistence'] = np.repeat(anc_all[:, None, :], HORIZON, axis=1)
print('\ndone:', list(test_pred))

dynamic  test RMSE 59.08  (nb14 7-seed was 58.79)

[Static-GCN v3] seed 42
    ep  4 train 0.1240 val 0.1386 (21s) best 0.1386
    ep  8 train 0.1042 val 0.1497 (20s) best 0.1386
    ep 12 train 0.0942 val 0.1457 (19s) best 0.1386
    ep 16 train 0.0890 val 0.1474 (19s) best 0.1386
  -> seed 42 test RMSE 60.34

[Static-GCN v3] seed 7
    ep  4 train 0.1223 val 0.1372 (19s) best 0.1372
    ep  8 train 0.1013 val 0.1532 (19s) best 0.1372
    ep 12 train 0.0927 val 0.1525 (20s) best 0.1372
    ep 16 train 0.0873 val 0.1589 (19s) best 0.1372
  -> seed 7 test RMSE 59.21

[Static-GCN v3] seed 123
    ep  4 train 0.1209 val 0.1412 (21s) best 0.1400
    ep  8 train 0.1014 val 0.1559 (19s) best 0.1390
    ep 12 train 0.0922 val 0.1577 (19s) best 0.1390
    ep 16 train 0.0875 val 0.1600 (19s) best 0.1390
    ep 17 train 0.0865 val 0.1541 (19s) best 0.1390
  -> seed 123 test RMSE 60.57

[Static-GCN v3] seed 2024
    ep  4 train 0.1217 val 0.1585 (19s) best 0.1408
    ep  8 train 0.1012 val 0.1548

In [9]:
ORDER = ['Persistence', 'No-graph v3', 'Static-GCN v3', 'WA-TGCN v3 (dynamic)']
ar = from_log(test_actual)

print("v2 pipeline (CPCB wind, nb11):  persistence 83.87 | no-graph 62.18 | static-GCN 61.73 | WA-TGCN 60.88")
print()
print(f"{'model':24}{'RMSE':>8}{'MAE':>8}{'R2':>8}")
for name in ORDER:
    pr = from_log(test_pred[name])
    print(f"{name:24}{np.sqrt(mean_squared_error(ar.ravel(), pr.ravel())):>8.2f}"
          f"{mean_absolute_error(ar.ravel(), pr.ravel()):>8.2f}"
          f"{r2_score(ar.ravel(), pr.ravel()):>8.3f}")

wt = sc(test_pred['WA-TGCN v3 (dynamic)'], test_actual)['rmse']
sg = sc(test_pred['Static-GCN v3'], test_actual)['rmse']
ng = sc(test_pred['No-graph v3'], test_actual)['rmse']
print(f"\nwind-aware graph vs static graph : {sg - wt:+.2f} ug/m3   (v2 pipeline this gap was +0.85)")
print(f"any graph vs no graph           : {ng - sg:+.2f} ug/m3")
print(f"no-graph v3 vs no-graph v2      : {62.18 - ng:+.2f} ug/m3  (isolates the ERA5-wind + forecast-wind feature effect)")

# by INPUT-window... no: by FORECAST-period ERA5 wind speed (does the wind graph help when transport is active?)
fut_spd = np.array([era5_spd[g+INPUT_WINDOW:g+INPUT_WINDOW+HORIZON].mean() for g in test_starts])
w33, w66 = np.percentile(fut_spd, [33, 66])
wind_reg = {f'Calm(<{w33:.1f})': fut_spd <= w33,
            'Breezy': (fut_spd > w33) & (fut_spd <= w66),
            f'Windy(>{w66:.1f})': fut_spd > w66}
print(f"\n{'model':24}" + "".join(f'{k:>14}' for k in wind_reg))
wr = {}
for name in ORDER:
    pr = from_log(test_pred[name]); r = {}
    for k, msk in wind_reg.items():
        r[k] = float(np.sqrt(mean_squared_error(ar[msk].ravel(), pr[msk].ravel())))
    wr[name] = r
    print(f"{name:24}" + "".join(f'{r[k]:>14.2f}' for k in wind_reg))
ks = list(wind_reg)
print("\nwind-aware advantage over static-GCN, by forecast-period wind:")
for k in ks:
    print(f"  {k:16} {wr['Static-GCN v3'][k] - wr['WA-TGCN v3 (dynamic)'][k]:+.2f} ug/m3")

v2 pipeline (CPCB wind, nb11):  persistence 83.87 | no-graph 62.18 | static-GCN 61.73 | WA-TGCN 60.88

model                       RMSE     MAE      R2
Persistence                83.87   56.93   0.306
No-graph v3                58.41   37.79   0.664
Static-GCN v3              58.16   38.16   0.667
WA-TGCN v3 (dynamic)       59.08   38.27   0.656

wind-aware graph vs static graph : -0.92 ug/m3   (v2 pipeline this gap was +0.85)
any graph vs no graph           : +0.25 ug/m3
no-graph v3 vs no-graph v2      : +3.77 ug/m3  (isolates the ERA5-wind + forecast-wind feature effect)

model                       Calm(<1.4)        Breezy   Windy(>1.9)
Persistence                     105.26         76.91         64.76
No-graph v3                      79.44         49.39         39.26
Static-GCN v3                    78.37         49.09         40.61
WA-TGCN v3 (dynamic)             80.48         49.61         39.89

wind-aware advantage over static-GCN, by forecast-period wind:
  Calm(<1.4)       -

In [10]:
out = {
    'note': 'RG6 ablation on the v3 pipeline (ERA5 wind + oracle forecast wind). Only the graph varies. Seeds 42/7/123/2024/999.',
    'v2_pipeline_ref': {'persistence': 83.87, 'no_graph': 62.18, 'static_gcn': 61.73, 'wa_tgcn': 60.88},
    'v3_pipeline': {name: {k: round(v, 2) for k, v in sc(test_pred[name], test_actual).items()} for name in ORDER},
    'wind_aware_vs_static_gap': round(sg - wt, 2),
    'by_forecast_wind_regime': {name: {k: round(v, 2) for k, v in wr[name].items()} for name in ORDER},
}
json.dump(out, open(os.path.join(RESULTS_DIR, 'baselines_v3.json'), 'w'), indent=2)
print('saved baselines_v3.json')

saved baselines_v3.json
